# Holiday Feature Engineering

This notebook generates holiday-related features from `holidays_events.csv` combined with `stores.csv` to accurately apply geographic scope (National / Regional / Local) to each store-date pair.

## Features Created
| Feature | Description |
|---|---|
| `is_national_holiday` | 1 if the date is a national holiday |
| `is_regional_holiday` | 1 if the date is a regional holiday matching the store's state |
| `is_local_holiday` | 1 if the date is a local holiday matching the store's city |
| `is_transferred_holiday` | 1 if the holiday was originally transferred |
| `holiday_type` | Numeric encoding of the holiday type |
| `is_carnaval_feature` | 1 if the holiday is Carnaval |
| `days_to_next_holiday` | Days until the next holiday |
| `days_after_last_holiday` | Days since the last holiday |
| `is_earthquake_period` | 1 for the month following the April 2016 Ecuador earthquake |

## Pipeline Overview
```
Load Data → Preprocess Dates → Filter Transferred Holidays
    → Encode Types → Flag Carnaval → Merge Geographic Scope
    → Apply Holiday Flags → Halo Effect → Earthquake Dummy → Output
```

## 1. Import Libraries

We use **pandas** for all data manipulation and **numpy** for numeric operations.

In [1]:
import pandas as pd
import numpy as np

## 2. Load Data

Three datasets are required:
- **`train.csv`** — the main transaction/sales history (must contain `date` and `store_nbr`).
- **`holidays_events_cleaned.csv`** — the cleaned holidays and events calendar.
- **`stores_cleaned.csv`** — store metadata containing `city` and `state` for geographic matching.

In [2]:
BASE_PATH = r'D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data'

df_train    = pd.read_csv(rf'{BASE_PATH}\processed\train_cleaned.csv')
df_holidays = pd.read_csv(rf'{BASE_PATH}\processed\holidays_events_cleaned.csv')
df_stores   = pd.read_csv(rf'{BASE_PATH}\processed\stores_cleaned.csv')

print(f'Train shape      : {df_train.shape}')
print(f'Holidays shape   : {df_holidays.shape}')
print(f'Stores shape     : {df_stores.shape}')
df_train.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Topic_13_Project\\Topic_13_Retail_Store_Sales_Time_Series\\data\\processed\\holidays_events_cleaned.csv'

## 3. Step 1 — Date Preprocessing

Convert the `date` column to `datetime` in both `main_df` and `holidays_events_df` to enable date-based operations (comparison, arithmetic, etc.).

In [ ]:
df_train['date']    = pd.to_datetime(df_train['date'])
df_holidays['date'] = pd.to_datetime(df_holidays['date'])

print('Date columns converted.')
print(f"Train date range   : {df_train['date'].min()} → {df_train['date'].max()}")
print(f"Holiday date range : {df_holidays['date'].min()} → {df_holidays['date'].max()}")

## 4. Step 2 — Handle Transferred Holidays

In Ecuador's holiday system a holiday can be **transferred**:
- A row with `transferred = True` means the original date is **cancelled** (employees work that day).
- The actual day off appears as a separate row with `type = 'Transfer'`.

We **keep only rows where `transferred == False`** so we model the actual days off.
We also create an `is_transferred` numeric flag to capture that the holiday was originally moved.

In [ ]:
df_holidays['is_transferred'] = df_holidays['transferred'].astype(int)
valid_holidays = df_holidays[df_holidays['transferred'] == False].copy()

print(f'Total holiday rows       : {len(df_holidays)}')
print(f'Valid (non-transferred)  : {len(valid_holidays)}')
print(f'Removed (transferred)    : {len(df_holidays) - len(valid_holidays)}')
valid_holidays[['date', 'type', 'locale', 'locale_name', 'transferred']].head()

## 5. Step 3 — Encode Holiday Type to Numeric

The `type` column is a string category. We map it to an ordered integer so that tree-based models can learn ordinal relationships between types.

| Type | Code | Meaning |
|---|---|---|
| Work Day | 0 | Not a holiday — employees work |
| Holiday | 1 | Standard public holiday |
| Event | 2 | Special event (non-statutory) |
| Additional | 3 | Extra holiday granted by decree |
| Transfer | 4 | The compensated day-off from a transferred holiday |
| Bridge | 5 | Bridge day between a holiday and weekend |

In [ ]:
type_mapping = {
    'Holiday'   : 1,
    'Event'     : 2,
    'Additional': 3,
    'Transfer'  : 4,
    'Bridge'    : 5,
    'Work Day'  : 0
}

valid_holidays['holiday_type_encoded'] = valid_holidays['type'].map(type_mapping).fillna(0)

print('Holiday type distribution after encoding:')
valid_holidays.groupby(['type', 'holiday_type_encoded']).size().reset_index(name='count')

## 6. Step 4 — Create Carnaval Feature

**Carnaval** is a major multi-day celebration in Ecuador with a pronounced and unique impact on retail sales (large crowds, travel, specific product spikes).  
We extract a dedicated binary flag by searching for the word `'Carnaval'` in the `description` column (case-insensitive).

In [ ]:
valid_holidays['is_carnaval'] = (
    valid_holidays['description']
    .str.contains('Carnaval', case=False, na=False)
    .astype(int)
)

carnaval_dates = valid_holidays[valid_holidays['is_carnaval'] == 1][['date', 'description', 'locale']]
print(f'Carnaval event rows found: {len(carnaval_dates)}')
carnaval_dates

## 7. Step 5 — Merge Geographic Scope

Holidays in Ecuador have three scopes:
- **National** — applies to every store across the country.
- **Regional** — applies only to stores in a specific *state* (`locale_name` = state name).
- **Local** — applies only to stores in a specific *city* (`locale_name` = city name).

We first join the main dataframe with `stores_df` to attach `city` and `state` to each store-date row, enabling the geographic filtering in the next step.

In [ ]:
df = df_train.merge(
    df_stores[['store_nbr', 'city', 'state']],
    on='store_nbr',
    how='left'
)

print(f'Merged dataframe shape : {df.shape}')
print(f'Unique stores          : {df["store_nbr"].nunique()}')
print(f'States in data         : {sorted(df["state"].unique())}')
df[['date', 'store_nbr', 'city', 'state']].drop_duplicates('store_nbr').head(5)

## 8. Step 6 — Build and Apply Holiday Flags

### 8.1 Pre-compute Masks

For each valid holiday row we compute a boolean mask over `df` based on its locale:
- **National** → all rows where `date == holiday_date`.
- **Regional** → rows where `date == holiday_date` **AND** `state == locale_name`.
- **Local** → rows where `date == holiday_date` **AND** `city == locale_name`.

We collect all masks first to avoid repeatedly iterating the full dataframe.

In [ ]:
holiday_features = []

for _, row in valid_holidays.iterrows():
    h_date       = row['date']
    h_locale     = row['locale']
    h_locale_name = row['locale_name']
    h_encoded    = row['holiday_type_encoded']
    is_transferred = row['is_transferred']
    is_carnaval  = row['is_carnaval']

    if h_locale == 'National':
        mask = (df['date'] == h_date)
    elif h_locale == 'Regional':
        mask = (df['date'] == h_date) & (df['state'] == h_locale_name)
    elif h_locale == 'Local':
        mask = (df['date'] == h_date) & (df['city'] == h_locale_name)
    else:
        continue

    holiday_features.append({
        'mask'         : mask,
        'locale'       : h_locale,
        'is_transferred': is_transferred,
        'encoded_type' : h_encoded,
        'is_carnaval'  : is_carnaval
    })

print(f'Total holiday mask entries built: {len(holiday_features)}')

### 8.2 Initialise Holiday Columns

All holiday flag columns are initialised to **0** (not a holiday) before we selectively set qualifying rows to 1.

In [ ]:
df['is_national_holiday']  = 0
df['is_regional_holiday']  = 0
df['is_local_holiday']     = 0
df['is_transferred_holiday'] = 0
df['holiday_type']         = 0
df['is_carnaval_feature']  = 0

print('Holiday columns initialised to 0:')
print(df[['is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
          'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature']].sum())

### 8.3 Assign Flag Values

We iterate over the pre-computed masks and use `.loc[]` to assign the correct flag values.  
If multiple holidays fall on the same date (e.g. both a National and Local event), the last assignment wins for `holiday_type` and `is_carnaval_feature`, but the scope flags accumulate independently.

In [ ]:
for h in holiday_features:
    if h['locale'] == 'National':
        df.loc[h['mask'], 'is_national_holiday'] = 1
    elif h['locale'] == 'Regional':
        df.loc[h['mask'], 'is_regional_holiday'] = 1
    elif h['locale'] == 'Local':
        df.loc[h['mask'], 'is_local_holiday'] = 1

    df.loc[h['mask'], 'is_transferred_holiday'] = h['is_transferred']
    df.loc[h['mask'], 'holiday_type']           = h['encoded_type']
    df.loc[h['mask'], 'is_carnaval_feature']    = h['is_carnaval']

print('Holiday flags assigned. Summary:')
print(df[['is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
          'is_transferred_holiday', 'is_carnaval_feature']].sum())

## 9. Step 7 — Halo Effect Features

Consumer behaviour changes in the **days leading up to** and **days following** a holiday ("halo effect"):
- **Pre-holiday surge**: shoppers stock up 1–3 days before.
- **Post-holiday dip**: reduced demand immediately after.

We compute two features:
- `days_to_next_holiday` — how many days until the next any-locale holiday.
- `days_after_last_holiday` — how many days have passed since the last any-locale holiday.

> **Optimisation**: We compute these values on the set of **unique dates** in `df` and then left-merge back, avoiding redundant computation for each store row on the same date.

In [ ]:
unique_holiday_dates = (
    valid_holidays['date']
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

def get_days_to_next(date):
    """Return the number of days until the next holiday, or -1 if none exists."""
    future = unique_holiday_dates[unique_holiday_dates > date]
    return (future.iloc[0] - date).days if not future.empty else -1

def get_days_after_last(date):
    """Return the number of days since the last holiday, or -1 if none exists."""
    past = unique_holiday_dates[unique_holiday_dates < date]
    return (date - past.iloc[-1]).days if not past.empty else -1

# Compute on unique dates only (much faster)
unique_dates = df[['date']].drop_duplicates().copy()
unique_dates['days_to_next_holiday']   = unique_dates['date'].apply(get_days_to_next)
unique_dates['days_after_last_holiday'] = unique_dates['date'].apply(get_days_after_last)

# Merge back to the main dataframe
df = df.merge(unique_dates, on='date', how='left')

print('Halo effect features added:')
print(df[['days_to_next_holiday', 'days_after_last_holiday']].describe())

## 10. Step 8 — Earthquake Dummy (April – May 2016)

A **magnitude 7.8 earthquake** struck Ecuador on **16 April 2016**, causing significant disruption to supply chains and consumer behaviour.  
The impact on retail sales was measurable for approximately **one month** after the event.

We capture this structural break with a binary dummy variable:
- `is_earthquake_period = 1` for dates in `[2016-04-16, 2016-05-16]`.
- `is_earthquake_period = 0` otherwise.

In [ ]:
earthquake_start = pd.to_datetime('2016-04-16')
earthquake_end   = pd.to_datetime('2016-05-16')

df['is_earthquake_period'] = (
    (df['date'] >= earthquake_start) & (df['date'] <= earthquake_end)
).astype(int)

n_earthquake_rows = df['is_earthquake_period'].sum()
print(f'Rows flagged as earthquake period : {n_earthquake_rows}')
print(f'Unique dates in earthquake period  : {df[df["is_earthquake_period"]==1]["date"].nunique()}')

## 11. Cleanup — Drop Temporary Columns

The `city` and `state` columns were added solely to enable geographic matching in Step 5–6.  
We remove them now so the output dataframe is clean and contains only the engineered features.

In [ ]:
df = df.drop(columns=['city', 'state'])

print(f'Final dataframe shape : {df.shape}')
print('\nColumns:')
print(df.columns.tolist())

## 12. Output — Preview & Summary

Final check of the engineered features — value counts and a sample of rows to verify correctness.

In [ ]:
holiday_cols = [
    'is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
    'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature',
    'days_to_next_holiday', 'days_after_last_holiday', 'is_earthquake_period'
]

print('=== Feature Summary ===')
print(df[holiday_cols].describe().T[['mean', 'min', 'max']])
print()
print('=== Sample rows with active holidays ===')
df[df['is_national_holiday'] == 1][['date', 'store_nbr'] + holiday_cols].head(10)

## 13. Save Output

Persist the feature-enriched dataframe to the processed data folder for use in downstream modelling notebooks.

In [ ]:
OUTPUT_PATH = rf'{BASE_PATH}\processed\train_holiday_features.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to: {OUTPUT_PATH}')
print(f'Shape    : {df.shape}')